In [2]:
import pandas as pd
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments
from transformers import EarlyStoppingCallback
import torch

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
#path = "./files/df_ml_good_with_features.csv"
path = "df_ml_good_with_features.csv"
df = pd.read_csv(path)
df_amyloid = df[df["class"] == 1]

print(df_amyloid.columns)

Index(['id', 'sequence', 'length', 'class', 'merge_id', 'a3vSA', 'AAT',
       'Na4vSS', 'THSA', 'nHS', 'TA', 'net_charge', 'hydrophobicity',
       'polar_fraction', 'nonpolar_fraction', 'acidic_fraction',
       'basic_fraction', 'aromatic_fraction', 'beta_propensity', 'seq_entropy',
       'proline_fraction'],
      dtype='object')


In [6]:
features_map = {
    'beta_propensity': 'bp',
    'proline_fraction': 'pf',
    'AAT': 'aat',
    'net_charge': 'nc',
    'TA': 'ta',
    'polar_fraction': 'pol',
    'a3vSA': 'sa'
}

def build_input(example):
    feat_str = "|".join([
        f"{features_map[f]}={example[f]:.2f}"
        for f in features_map
    ])
    return feat_str + " <SEP> " + example["sequence"] + " <END>"

df_amyloid["text"] = df_amyloid.apply(build_input, axis=1)

/tmp/ipykernel_2344/3023121205.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_amyloid["text"] = df_amyloid.apply(build_input, axis=1)


In [7]:
dataset = Dataset.from_pandas(df_amyloid[['text']])
dataset = dataset.train_test_split(test_size=0.1, seed=42)

train_dataset = dataset["train"]
val_dataset = dataset["test"]

In [9]:
print((train_dataset[0]["text"]))

bp=0.92|pf=0.00|aat=1.43|nc=-1.90|ta=-5.18|pol=0.18|sa=-0.26 <SEP> QARDSGEIYAAAGDMHI <END>


In [8]:
tokenizer = AutoTokenizer.from_pretrained("nferruz/ProtGPT2")

tokenizer.add_special_tokens({
    "additional_special_tokens": ["<SEP>", "<END>"],
})

tokenizer.pad_token = tokenizer.eos_token

tokenizer.model_max_length = 128

model = AutoModelForCausalLM.from_pretrained("nferruz/ProtGPT2")

model.resize_token_embeddings(len(tokenizer))

lengths = [
    len(tokenizer(x)["input_ids"])
    for x in df_amyloid["text"].sample(1000)
]

print(max(lengths), sum(lengths)/len(lengths))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/850 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/357 [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/437 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: nferruz/ProtGPT2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...35}.attn.masked_bias | UNEXPECTED |  | 
transformer.h.{0...35}.attn.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`
The new lm_head weights will be initialized from a multivariate normal distr

81 69.486


In [10]:
print(tokenizer)

GPT2Tokenizer(name_or_path='nferruz/ProtGPT2', vocab_size=50257, model_max_length=128, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<|endoftext|>', 'eos_token': '<|endoftext|>', 'unk_token': '<|endoftext|>', 'pad_token': '<|endoftext|>'}, added_tokens_decoder={
	0: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
	50257: AddedToken("<SEP>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	50258: AddedToken("<END>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)


In [11]:
def tokenize_function(example):
    tokens = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=100
    )

    tokens["labels"] = [
        -100 if t == tokenizer.pad_token_id else t
        for t in tokens["input_ids"]
    ]
    return tokens


train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/905 [00:00<?, ? examples/s]

Map:   0%|          | 0/101 [00:00<?, ? examples/s]

In [12]:
for i in range(5):
    print(train_dataset[i])

{'text': 'bp=0.92|pf=0.00|aat=1.43|nc=-1.90|ta=-5.18|pol=0.18|sa=-0.26 <SEP> QARDSGEIYAAAGDMHI <END>', '__index_level_0__': 380, 'input_ids': [66, 80, 29, 16, 14, 25, 18, 92, 80, 70, 29, 16, 14, 16, 16, 92, 65, 65, 84, 29, 17, 14, 20, 19, 92, 78, 67, 29, 13, 17, 14, 25, 16, 92, 84, 65, 29, 13, 21, 14, 17, 24, 92, 80, 79, 76, 29, 16, 14, 17, 24, 92, 83, 65, 29, 13, 16, 14, 18, 22, 221, 50257, 221, 1156, 36, 784, 399, 1888, 594, 553, 221, 50258, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'labels': [66, 80, 29, 16, 14, 25, 18, 92, 80, 70, 29, 16, 14, 16, 16, 92, 65, 65, 84, 29, 17, 14, 20, 19, 92, 78, 67, 29, 13, 17, 14, 25, 16, 92, 

In [11]:
#zamrażanie warstw - bardzo mało danych
for param in model.transformer.h[:-6].parameters():
    param.requires_grad = False

In [12]:
training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/protGPT2-results",
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    learning_rate=2e-5,
    num_train_epochs=50,
    do_eval=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    logging_steps=50,
    fp16=True, #gpu
    weight_decay=0.01, #L2
)

model.config.resid_pdrop = 0.2
model.config.embd_pdrop = 0.2
model.config.attn_pdrop = 0.2

In [13]:
print("vocab:", len(tokenizer))

max_id = max(max(x) for x in train_dataset["input_ids"])
print("max input id:", max_id)

vocab: 50259
max input id: 50258


In [14]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

In [15]:
print("vocab:", len(tokenizer))

max_id = max(max(x) for x in train_dataset["input_ids"])
print("max input id:", max_id)

vocab: 50259
max input id: 50258


In [16]:
trainer.train()

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,2.455797,2.263177
2,1.961521,1.749809
3,1.581812,1.311126
4,1.350214,1.181260
5,1.186258,1.102196
6,1.072617,1.045793
7,1.038087,0.986142
8,1.023912,0.977075
9,0.921070,0.959593
10,0.901725,0.951865


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


TrainOutput(global_step=8607, training_loss=1.2879628541786192, metrics={'train_runtime': 2976.7552, 'train_samples_per_second': 15.201, 'train_steps_per_second': 7.609, 'total_flos': 7308463756800000.0, 'train_loss': 1.2879628541786192, 'epoch': 19.0})

## Generowanie

In [13]:
features_map = {
    'beta_propensity': 'bp',
    'proline_fraction': 'pf',
    'AAT': 'aat',
    'net_charge': 'nc',
    'TA': 'ta',
    'polar_fraction': 'pol',
    'a3vSA': 'sa'
}

def perturb_features(row, noise_level=0.05):
    new_feats = {}
    for f in features_map:
        val = row[f]
        noise = np.random.normal(0, noise_level * abs(val) if val != 0 else noise_level)
        new_feats[f] = val + noise
    return new_feats

In [14]:
def build_prompt(feats):
    return "|".join([
        f"{features_map[f]}={feats[f]:.2f}"
        for f in features_map
    ]) + " <SEP>"

In [15]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_path = "/content/drive/MyDrive/protGPT2-results/checkpoint-7248"

tokenizer = AutoTokenizer.from_pretrained("nferruz/ProtGPT2")
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(model_path).to(device)
model.eval()

Loading weights:   0%|          | 0/436 [00:10<?, ?it/s]

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50259, 1280)
    (wpe): Embedding(1024, 1280)
    (drop): Dropout(p=0.2, inplace=False)
    (h): ModuleList(
      (0-35): 36 x GPT2Block(
        (ln_1): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=3840, nx=1280)
          (c_proj): Conv1D(nf=1280, nx=1280)
          (attn_dropout): Dropout(p=0.2, inplace=False)
          (resid_dropout): Dropout(p=0.2, inplace=False)
        )
        (ln_2): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=5120, nx=1280)
          (c_proj): Conv1D(nf=1280, nx=5120)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.2, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=1280, out_features=50259, bias=False)
)

In [23]:
import pandas as pd
import numpy as np
import re

df = pd.read_csv("df_ml_good_with_features.csv")

generated = []

end_token_id = tokenizer.convert_tokens_to_ids("<END>")

N_PER_ROW = 2   # ile generacji na jedną sekwencję
BATCH_SIZE = 10

for _, row in df.sample(1000).iterrows():

    for _ in range(N_PER_ROW):

        feats = perturb_features(row, noise_level=0.1)
        prompt = build_prompt(feats)

        inputs = tokenizer(prompt, return_tensors="pt").to(device)

        with torch.no_grad():
            outputs = model.generate(
            **inputs,
            max_new_tokens=50,   # długość samej sekwencji
            min_new_tokens=5,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            top_k=30,
            repetition_penalty=1.5,
            no_repeat_ngram_size=3,
            num_return_sequences=BATCH_SIZE,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=end_token_id,
        )

        seqs = tokenizer.batch_decode(outputs, skip_special_tokens=True)

        for s in seqs:
            if "<SEP>" in s:
                s = s.split("<SEP>")[-1]

            s = re.sub(r"[^ACDEFGHIKLMNPQRSTVWY]", "", s)

            if len(s) >= 5:
                generated.append(s)


print("Generated:", len(generated))

Generated: 17709


In [25]:
df_out = pd.DataFrame({
    "sequence": generated,
    "length": [len(s) for s in generated]
})

df_out.to_csv("/content/drive/MyDrive/protGPT2-results/protGPT2_generated_sequences-17709.csv", index=False)

print("Generated:", len(generated))
print("Saved to: generated_sequences.csv")

Generated: 17709
Saved to: generated_sequences.csv


In [26]:
for i in range(10):
    print(f"{i} {generated[i]}")

0 LFFEDVN
1 GDRSRVFIGNV
2 VQIINNIRDITKDEVTQHFTVHNGNV
3 SFGSARAEGHGPVVQAGDIVLGGLGSLVYGQMNEPPGNRLRVALTGLTM
4 GRYSFVIQGRDFS
5 TGNFLINQNFAEDA
6 SSQHYSFQGQNIQGTGNVYTGDHNTQHNTVNIGDQSTINTNNSVTINNNNNSYSANNYTVHIGNNTFHIGGAA
7 GSHTFHGPVALVGDN
8 GNTFVNISGGNFGSVGNV
9 GSTFHGPVVQGDV
